# Cognitive Wi-Fi — PPO Single-AP Training v2 (12D Observation / 3D Action)

**Yêu cầu 1/2.** Notebook thay thế pipeline cũ nhưng **không** thay đổi sản phẩm nghiên cứu Multi-AP. Lưu ý đây là Digital Twin **synthetic, chưa hiệu chuẩn**, KHÔNG được coi là bằng chứng tác tử tối ưu mạng Wi-Fi thực tế.

> Dữ liệu phần cứng do tác giả xác nhận: Xiaomi AX3000T hiện chỉ kiểm chứng được **channel, bandwidth, Tx power**. MCS/Beacon/OBSS_PD/Beamforming chưa kiểm chứng, nên mặc định KHÔNG học điều khiển bốn tham số đó. Kết quả `best_model_v2_UNCALIBRATED.zip` chỉ dành cho nghiên cứu/shadow mode.

**Không tương thích checkpoint cũ**: `best_model (2).zip` nhận 8D/7D, bản mới nhận 12D/3D. Chạy các cell theo thứ tự; cell TRAIN không tự chạy nếu chưa bật cờ.


## 0. Cài đặt Dependencies

Chạy trong Colab/WSL2. Khi dùng WSL2 ngoài notebook, thay `%pip` bằng `python -m pip install ...`. Bạn cần Python 3.10 trở lên.

In [ ]:
%pip install -q "gymnasium>=1.0,<1.4" "stable-baselines3>=2.4,<3" "numpy>=1.26,<3" "matplotlib>=3.7,<4" "tensorboard>=2.14,<3"

## 1. Source of Truth — Digital Twin 12D/3D

Cú pháp `%%writefile` tạo file Python thật từ cell, tránh sai lệch giữa môi trường Colab và code trong repo. **Không** tự động tạo lưu lượng jam thực, không có lệnh SSH hay thay đổi router.

In [ ]:
%%writefile wifi_rl_core_v2.py
"""Cognitive Wi-Fi v2: SINGLE-AP, OFFLINE, SYNTHETIC controller research environment.

IMPORTANT: This is a deliberately simplified, uncalibrated *research simulator*,
NOT an IEEE 802.11 PHY/MAC emulator and NOT suitable to justify autonomous
configuration changes on production routers.

The safe deployable action subset is [primary_channel, channel_width, tx_power].
Four other legacy actions are explicitly NOT enabled until hardware validation.
"""
from __future__ import annotations
from dataclasses import dataclass
from typing import Any
import math
import numpy as np

try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    gym = None
    spaces = None

OBS8_NAMES = (
    'avg_rssi', 'noise_floor', 'sta_count', 'retry_rate', 'p_c',
    'airtime_util', 'ofdma_ratio', 'is_break_time',
)
OBS12_NAMES = OBS8_NAMES + ('current_channel_idx', 'current_bw_idx',
                           'current_tx_idx', 'dwell_time_fraction')
ACTION3_NAMES = ('channel_idx', 'bandwidth_idx', 'tx_power_idx')
ACTION3_NVECS = np.array([3, 3, 7], dtype=np.int64)
CHANNELS_5G = np.array([36, 40, 44], dtype=np.int64)
WIDTHS_MHZ = np.array([20, 40, 80], dtype=np.int64)
TX_POWERS_DBM = np.arange(14, 21, dtype=np.int64)
# Legacy action mapping for *display / shadow-mode* only.
# The four unsupported dimensions stay fixed and are NOT advertised as verified.
LEGACY_SAFE_UNVERIFIED_INDEX = np.array([7, 1, 0, 0], dtype=np.int64)  # mcs, beacon, obss_pd, bf
# Two spatial streams, approximately nominal HE 20 MHz GI 0.8us rates (Mbps);
# values are approximate anchors, NOT a calibrated link predictor.
HE20_TWO_NSS_Mbps = np.array([17.2,34.4,51.6,68.8,103.2,137.6,
                             154.9,172.1,206.5,229.4,258.1,286.8], dtype=float)
SINR_REQ_DB = np.array([1,3,5,7,10,13,16,19,22,25,28,31], dtype=float)


def covered_20mhz(channel_index: int, bandwidth_index: int) -> tuple[int, ...]:
    """For the chosen primary in the 36-48 block, return occupied 20MHz bins.

    CH36/40/44 as HE80 choices occupy exactly the same four bins.
    Returns channel IDs, not 'orthogonal channel numbers'.
    """
    ch = int(CHANNELS_5G[int(channel_index)])
    width = int(WIDTHS_MHZ[int(bandwidth_index)])
    if width == 20:
        return (ch,)
    if width == 40:
        return (36, 40) if ch in (36, 40) else (44, 48)
    return (36, 40, 44, 48)


def _dbm_from_linear_mw(mw: float) -> float:
    return 10.0 * math.log10(max(1e-20, mw))


def _linear_mw(dbm: float) -> float:
    return 10.0 ** (dbm / 10.0)


@dataclass
class RadioParams:
    dt_s: float = 10.0
    episode_steps: int = 120
    queue_limit_mbit: float = 12.0
    mac_efficiency: float = 0.68
    radio_rate_scale: float = 1.0
    # These are sensitivity parameters, not measurement-calibrated constants.
    packet_loss_slope: float = 0.18
    swap_outage_s: float = 1.0
    burden_txpower: float = 0.015
    obs_noise_rssi_std: float = 0.7
    validate_mode: str = 'train'  # 'train', 'validation', 'ood'


class SingleAPRadioCore:
    """Action-dependent Markov-state simulator with exogenous, correlated loads.

    Only the local AP is simulated; external channel occupancy is generated as
    a stochastic process. No claim to multi-AP centralized optimization.
    """
    def __init__(self, params: RadioParams | None = None):
        self.params = params or RadioParams()
        self.rng = np.random.default_rng(0)
        self.t = 0
        self.config = np.array([0, 2, 3], dtype=np.int64)
        self.time_on_config = 100
        self.offer_mbps = 50.0
        self.client_count = 16
        self.rssi_base_dbm_at17 = -61.0
        self.noise_dbm = np.full(4, -95.0)
        self.channel_occupancy = np.full(4, .10)
        self.queue_mbit = 0.0
        self.break_time = 0
        self.metrics: dict[str, float] = {}
        self.regime = 0
        self.phase_length = 0

    def reset(self, seed: int, params: RadioParams | None = None) -> dict[str, float]:
        if params is not None:
            self.params = params
        self.rng = np.random.default_rng(int(seed))
        self.t = 0
        self.config = np.array([0, 2, 3], dtype=np.int64)
        self.time_on_config = 100
        self.queue_mbit = 0.0
        self.regime = int(self.rng.choice([0,1,2],p=[.5,.3,.2]))
        self.phase_length = int(self.rng.integers(8, 26))
        self.rssi_base_dbm_at17 = float(self.rng.uniform(-75,-49))
        self.client_count = int(self.rng.integers(6,45))
        self.offer_mbps = float(self.rng.uniform(20,100))
        self.noise_dbm = self.rng.uniform(-99, -89, size=4)
        self.channel_occupancy = self.rng.uniform(.04, .23, size=4)
        self.break_time = 0
        self.metrics = self._calculate_observables_initial()
        return self.metrics.copy()

    def _calculate_observables_initial(self) -> dict[str, float]:
        covered = covered_20mhz(int(self.config[0]),int(self.config[1]))
        ids=[(ch-36)//4 for ch in covered]
        noise=float(np.mean(self.noise_dbm[ids]))
        return dict(avg_rssi=float(self.rssi_base_dbm_at17),
                    noise_floor=noise, sta_count=float(self.client_count),
                    retry_rate=.08, p_c=.05,
                    airtime_util=float(np.mean(self.channel_occupancy[ids])),
                    ofdma_ratio=float(np.clip(.75-.003*self.client_count,0,1)),
                    is_break_time=float(self.break_time),
                    throughput_mbps=0.0, latency_p99_ms=3.0, loss_rate=0.0,
                    uqs_proxy=1.0, offered_mbps=float(self.offer_mbps),
                    queue_mbit=0.0, capacity_mbps=0.0)

    def observe(self) -> np.ndarray:
        """Return normalized 12D observation, with stable train/inference scaling."""
        m=self.metrics
        base=np.array([m[k] for k in OBS8_NAMES],dtype=np.float32)
        low=np.array([-90,-105,0,0,0,0,0,0],np.float32)
        high=np.array([-30,-50,300,1,1,1,1,1],np.float32)
        x=(np.clip(base,low,high)-low)/(high-low)
        extras=[float(self.config[0]/2),float(self.config[1]/2),
                float(self.config[2]/6),float(min(self.time_on_config/12,1.0))]
        return np.clip(np.array([*x,*extras],dtype=np.float32),0,1)

    def _advance_exogenous(self) -> None:
        r=self.rng
        self.t += 1
        self.phase_length -= 1
        if self.phase_length <= 0:
            # stochastic regime switching, not a fixed four-phase curriculum
            p=[.51,.27,.22] if self.params.validate_mode=='train' else [0.34,.34,.32]
            if self.params.validate_mode=='ood': p=[.17,.33,.50]
            self.regime=int(r.choice([0,1,2],p=p))
            self.phase_length=int(r.integers(5,25))
        target_sta=[18,55,135][self.regime]
        target_load=[40,130,300][self.regime]
        if self.params.validate_mode=='ood':
            target_sta= int(target_sta*1.3)
            target_load= int(target_load*1.7)
        self.client_count=int(np.clip(.82*self.client_count+.18*target_sta+r.normal(0,5),1,300))
        self.offer_mbps=float(np.clip(.78*self.offer_mbps+.22*target_load+r.normal(0,22),2,700))
        # correlated per-channel interference time series; independent shocks
        target_busy=np.array([.10,.14,.12,.08])
        if self.regime == 1:
            target_busy[int(r.integers(0,3))] = .70
        elif self.regime == 2:
            target_busy[r.choice(4,size=2,replace=False)] = .76
        self.channel_occupancy=np.clip(.78*self.channel_occupancy +.22*target_busy+
                                       r.normal(0,.035,size=4), 0,.98)
        target_noise=-97.0+26.0*self.channel_occupancy
        self.noise_dbm=np.clip(.75*self.noise_dbm+.25*target_noise+
                               r.normal(0,1.2,size=4),-105,-55)
        self.rssi_base_dbm_at17=float(np.clip(self.rssi_base_dbm_at17 +
                                             r.normal(0,.45), -87,-39))
        self.break_time=int(((self.t//36)%3)==2)

    def transition(self, action: np.ndarray | list[int]) -> tuple[dict[str,float],float,dict[str,float]]:
        a=np.asarray(action,dtype=np.int64).reshape(-1)
        if a.shape != (3,) or np.any(a<0) or np.any(a>=ACTION3_NVECS):
            raise ValueError('Action must be 3 indexes within MultiDiscrete([3,3,7])')
        old=self.config.copy()
        changed=(a!=old)
        channel_change=bool(changed[0]);width_change=bool(changed[1]);tx_change=bool(changed[2])
        # A channel/width reconfiguration consumes some service time; this is a
        # tunable assumption, NOT a device-verified outage time.
        outage=self.params.swap_outage_s if (channel_change or width_change) else 0.0
        self.config=a.copy()
        self.time_on_config = 0 if np.any(changed) else self.time_on_config+1
        self._advance_exogenous()
        bin_channels=covered_20mhz(int(a[0]),int(a[1]))
        ids=np.array([(c-36)//4 for c in bin_channels],dtype=int)
        width=int(WIDTHS_MHZ[int(a[1])]);tx_power=int(TX_POWERS_DBM[int(a[2])])
        # power summation across 20MHz bins; high bandwidth aggregates noise
        # station dump signal is received UL RSSI; AP Tx power chiefly affects DL
        # signal, not measured station uplink RSSI. Keep these distinct.
        uplink_rssi_dbm=float(np.clip(self.rssi_base_dbm_at17,-90,-30))
        downlink_rssi_dbm=float(np.clip(self.rssi_base_dbm_at17+tx_power-17.0,-90,-30))
        noise_mw=float(sum(_linear_mw(float(n)) for n in self.noise_dbm[ids]))
        noise_floor=_dbm_from_linear_mw(noise_mw)
        sinr=downlink_rssi_dbm-noise_floor
        mcs_idx=int(np.clip(np.searchsorted(SINR_REQ_DB,sinr,side='right')-1,0,11))
        phy_rate=float(HE20_TWO_NSS_Mbps[mcs_idx]*(width/20.0)*self.params.radio_rate_scale)
        busy=float(np.mean(self.channel_occupancy[ids]))
        competing=busy
        # collision proxy grows smoothly with associated stations and occupancy
        collision=float(np.clip(.015 + .28*(self.client_count/(self.client_count+55.0))+
                                .20*competing, 0, .75))
        phy_err=float(np.clip((SINR_REQ_DB[mcs_idx]+2-sinr)/20.0,0,.15))
        radio_service=max(1.,phy_rate*self.params.mac_efficiency*(1-competing)*
                          (1-collision)*(1-phy_err))
        radio_service*=max(0.,1-outage/self.params.dt_s)
        # Finite queue with arrivals and service in Mbit per decision interval
        arrivals=self.offer_mbps*self.params.dt_s
        available=self.queue_mbit+arrivals
        serviced=min(available,radio_service*self.params.dt_s)
        backlogged=max(0.,available-serviced)
        overflow=max(0.,backlogged-self.params.queue_limit_mbit)
        self.queue_mbit=float(backlogged-overflow)
        # End-to-end delivered = service less PHY residual loss; never exceeds offered+backlog
        delivered=max(0.,serviced*(1-phy_err))
        achieved_goodput=float(delivered/self.params.dt_s)
        step_drop=float(overflow + serviced*phy_err)
        drop_fraction=float(np.clip(step_drop/max(1e-9,arrivals),0,1))
        utilization=float(np.clip(self.offer_mbps/max(radio_service,1e-6),0,3))
        # Synthetic per-packet delays (not a driver-measured p99)
        queue_ms=(self.queue_mbit/max(1.,radio_service))*1000.0
        base_delay=2.0 + queue_ms + (outage*1000)/max(1.,self.params.dt_s)
        # fixed sample count ensures comparisons have consistent sampling effort
        samples=np.clip(base_delay+self.rng.lognormal(mean=.35,sigma=.38,size=64),0,5000)
        p99=float(np.percentile(samples,99))
        uqs=float(np.clip(min(self.offer_mbps,achieved_goodput)/max(1.,self.offer_mbps),0,1))
        rate_retry=float(np.clip(collision+phy_err+.005*self.queue_mbit,0,1))
        ofdma=float(np.clip(.70-.20*utilization-.001*self.client_count+
                             self.rng.normal(0,.03),0,1))
        # noisy survey measurement proxies for PHY/MAC; label as synthetic
        m={
          'avg_rssi':float(np.clip(uplink_rssi_dbm+self.rng.normal(0,self.params.obs_noise_rssi_std),-90,-30)),
          'noise_floor':float(np.clip(noise_floor+self.rng.normal(0,.7),-105,-50)),
          'sta_count':float(self.client_count),
          'retry_rate':rate_retry,
          'p_c':collision,
          'airtime_util':float(np.clip(max(competing,utilization*.70),0,1)),
          'ofdma_ratio':ofdma,'is_break_time':float(self.break_time),
          'throughput_mbps':achieved_goodput,
          'latency_p99_ms':p99,'loss_rate':drop_fraction,'uqs_proxy':uqs,
          'offered_mbps':float(self.offer_mbps),'queue_mbit':float(self.queue_mbit),
          'capacity_mbps':float(radio_service),
          'switch_outage_s':float(outage),
        }
        self.metrics=m
        # Reward is a prospective candidate, not the research utility U(X).
        reward=(1.3*uqs + 0.5*min(achieved_goodput/200.0,1.5)
                -1.7*drop_fraction -0.8*min(p99/100.0,5.0)
                -.25*channel_change -.10*width_change -.025*tx_change
                -self.params.burden_txpower*(tx_power-14)/6)
        # SLA uses *sampled synthetic* latency, not real network P99.
        info={
           'reward':float(reward), 'sla_pass':float(p99<=15 and drop_fraction<.01),
           'changed_radio':float(channel_change or width_change),
           'changed_tx':float(tx_change),
        }
        return m.copy(),float(reward),info


class CognitiveSingleAPEnv(gym.Env if gym is not None else object):
    """Gymnasium adapter, OBS 12D / ACTION 3D. NOT compatible with old 8D/7D weights."""
    metadata = {'render_modes': []}
    def __init__(self, params: RadioParams | None = None):
        if gym is None:
            raise ImportError('pip install gymnasium stable-baselines3')
        super().__init__()
        self.params=params or RadioParams()
        self.core=SingleAPRadioCore(self.params)
        self.observation_space=spaces.Box(low=0.0,high=1.0,shape=(12,),dtype=np.float32)
        self.action_space=spaces.MultiDiscrete(ACTION3_NVECS)
        self.elapsed=0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is None:
            seed=int(self.np_random.integers(0,2**31-1))
        self.core.reset(seed=int(seed))
        self.elapsed=0
        return self.core.observe(), {'model_type':'synthetic_uncalibrated_v2'}

    def step(self, action):
        m,r,info=self.core.transition(action)
        self.elapsed+=1
        terminated=False
        truncated=bool(self.elapsed>=self.params.episode_steps)
        return self.core.observe(), float(r),terminated,truncated,dict(info,**m)

    def render(self):
        return None


def encode_shadow_observation(raw_obs8: np.ndarray, current_action3: np.ndarray,
                              dwell_cycles: int) -> np.ndarray:
    """Reproduce the exact 12D train-time scaling at inference."""
    raw=np.asarray(raw_obs8,dtype=np.float32).reshape(-1)
    a=np.asarray(current_action3,dtype=np.int64).reshape(-1)
    if raw.shape!=(8,) or a.shape!=(3,) or np.any(a<0) or np.any(a>=ACTION3_NVECS):
        raise ValueError('Expected 8 sensor values and applied 3D configuration')
    low=np.array([-90,-105,0,0,0,0,0,0],np.float32)
    high=np.array([-30,-50,300,1,1,1,1,1],np.float32)
    if np.any(~np.isfinite(raw)) or np.any(raw<low) or np.any(raw>high):
        raise ValueError('Invalid raw sensor values')
    out=np.concatenate([(raw-low)/(high-low),
        np.array([a[0]/2,a[1]/2,a[2]/6,min(max(0,dwell_cycles)/12,1.0)],np.float32)])
    return out.astype(np.float32)


def heuristic_baseline(obs: np.ndarray) -> np.ndarray:
    """Conservative comparator with ONLY current-channel telemetry (no oracle info)."""
    rssi, noise, sta, retry, pc, busy, ofdma, brk = np.asarray(obs)[:8]
    # If alternative-channel occupancy is unavailable, do not assert CH40/44 cleaner.
    ch=0
    bw=0 if (busy>.70 or pc>.35) else 2 if busy<.30 else 1
    tx=5 if rssi < -68 else 3 if rssi < -57 else 1
    return np.array([ch,bw,tx],dtype=np.int64)


def fixed_baseline(obs: np.ndarray) -> np.ndarray:
    return np.array([0,2,3],dtype=np.int64)


def safe_legacy7_from3(action: np.ndarray | list[int]) -> np.ndarray:
    """Bridge for reading only; NOT authorization to send unsupported commands."""
    x=np.asarray(action,dtype=np.int64).reshape(-1)
    if x.shape!=(3,) or np.any(x<0) or np.any(x>=ACTION3_NVECS):
        raise ValueError('Invalid 3D action')
    return np.array([x[0],x[1],x[2],*LEGACY_SAFE_UNVERIFIED_INDEX],dtype=np.int64)


def validate_telemetry_snapshot(snapshot: dict, *, stale_seconds: float | None=None) -> dict:
    """Reject missing data and highlight proxy flags instead of inventing values."""
    from collections.abc import Mapping
    if not isinstance(snapshot,Mapping):
        raise TypeError('Telemetry must be a mapping')
    m={}
    for k in OBS8_NAMES:
        if k not in snapshot or snapshot[k] is None:
            raise ValueError(f'Missing live field {k}; never impute fake PHY values')
        v=float(snapshot[k]);
        if not np.isfinite(v):
            raise ValueError(f'Invalid {k}={snapshot[k]}')
        m[k]=v
    arr=np.array([m[k] for k in OBS8_NAMES],np.float32)
    low=np.array([-90,-105,0,0,0,0,0,0],np.float32)
    high=np.array([-30,-50,300,1,1,1,1,1],np.float32)
    if np.any(arr<low) or np.any(arr>high):
        raise ValueError('Observation out of training bounds: do not silently clip')
    alerts=[]
    for k in ('noise_floor','ofdma_ratio'):
        if snapshot.get(f'{k}_measured') is not True:
            alerts.append(f'{k} provenance not independently confirmed')
    if stale_seconds is None:
        alerts.append('snapshot freshness unspecified')
    elif stale_seconds>10:
        alerts.append(f'snapshot is stale ({stale_seconds:.1f}s)')
    return {'observation8':arr,'alerts':alerts,'shadow_only':True}


## 2. Training / Validation / Held-out OOD Benchmark

- Đánh giá độc lập được chia thành **validation** để chọn model và **held-out OOD** để kiểm tra cuối.
- Báo cáo khoảng tin cậy bootstrap theo episode. Các kết quả là **số liệu mô phỏng**, không thay thế HIL throughput/latency/loss.
- Không đưa tên `Oracle` cho baseline giả định.

In [ ]:
%%writefile train_evaluate_v2.py
"""Offline training / validation / benchmarking. No SSH or real router commands.

Run after installing requirements_v2.txt. Synthetic metrics are NOT measured KPIs.
"""
from __future__ import annotations
import argparse
import csv
import json
from pathlib import Path
from statistics import mean
import sys
import numpy as np
from wifi_rl_core_v2 import (CognitiveSingleAPEnv,RadioParams,OBS12_NAMES,
                            ACTION3_NAMES,ACTION3_NVECS,fixed_baseline,heuristic_baseline)


def _require_sb3():
    try:
        from stable_baselines3 import PPO
        from stable_baselines3.common.callbacks import EvalCallback
        from stable_baselines3.common.env_util import make_vec_env
        from stable_baselines3.common.monitor import Monitor
        from stable_baselines3.common.env_checker import check_env
        return PPO,EvalCallback,make_vec_env,Monitor,check_env
    except ImportError as e:
        raise RuntimeError('Install requirements_v2.txt first: pip install -r requirements_v2.txt') from e


def run_one_episode(policy, seed:int, mode='validation', episode_steps=120):
    """Return per-step metrics on paired deterministic synthetic seeds."""
    PPO,*_=_require_sb3()
    env=CognitiveSingleAPEnv(RadioParams(episode_steps=episode_steps,validate_mode=mode))
    obs,_=env.reset(seed=seed)
    rows=[];finished=False
    while not finished:
        if isinstance(policy,str):
            raw=np.array([env.core.metrics[k] for k in (
                'avg_rssi','noise_floor','sta_count','retry_rate','p_c',
                'airtime_util','ofdma_ratio','is_break_time')],np.float32)
            if policy=='fixed': action=fixed_baseline(raw)
            elif policy=='heuristic': action=heuristic_baseline(raw)
            else: raise ValueError(policy)
        else:
            action,_=policy.predict(obs,deterministic=True)
        obs,reward,terminated,truncated,info=env.step(action)
        rows.append({k:float(info[k]) for k in
            ('reward','sla_pass','changed_radio','changed_tx','throughput_mbps',
             'latency_p99_ms','loss_rate','uqs_proxy','offered_mbps',
             'queue_mbit','capacity_mbps')})
        finished=terminated or truncated
    env.close()
    return rows


def evaluate_benchmark(policies:dict, *, seeds, mode='validation',
                       episode_steps=120, export=None):
    """Held-out paired-seed scenarios, includes simple per-episode summary CIs.
    95% CIs are percentile bootstrap across episodes, not across correlated steps.
    """
    output=[]
    for name,policy in policies.items():
        for seed in seeds:
            rr=run_one_episode(policy,seed,mode,episode_steps)
            d={'policy':name,'seed':int(seed),'mode':mode}
            for k in rr[0]:
                d[k]=float(np.mean([row[k] for row in rr]))
            d['sla_all_steps_pass']=float(all(row['sla_pass']>0.5 for row in rr))
            output.append(d)
    if export is not None:
        path=Path(export);path.parent.mkdir(parents=True,exist_ok=True)
        with path.open('w',newline='',encoding='utf-8') as fh:
            writer=csv.DictWriter(fh,fieldnames=list(output[0].keys()))
            writer.writeheader(); writer.writerows(output)
    return output


def bootstrap_ci(rows, key, seed=2027, n_boot=1000):
    vals=np.array([r[key] for r in rows],dtype=float)
    rng=np.random.default_rng(seed)
    picks=rng.choice(vals,size=(n_boot,len(vals)),replace=True)
    return tuple(float(v) for v in np.quantile(picks.mean(axis=1),[.025,.975]))


def summarize(rows):
    summary={}
    for name in sorted({r['policy'] for r in rows}):
        subset=[x for x in rows if x['policy']==name]
        metrics={}
        for k in ('reward','throughput_mbps','latency_p99_ms','loss_rate','uqs_proxy',
                  'sla_pass','changed_radio','sla_all_steps_pass'):
            metrics[k]={'mean':float(np.mean([r[k] for r in subset])),
                        'ci95':bootstrap_ci(subset,k)}
        summary[name]=metrics
    return summary


def train_one(seed:int, timesteps:int, directory:Path, n_envs:int=4):
    PPO,EvalCallback,make_vec_env,Monitor,check_env=_require_sb3()
    out=directory/f'seed_{seed}'
    out.mkdir(parents=True,exist_ok=True)
    check_env(CognitiveSingleAPEnv(RadioParams()),warn=True)
    def factory():
        return CognitiveSingleAPEnv(RadioParams(validate_mode='train'))
    env=make_vec_env(factory,n_envs=n_envs,seed=seed)
    eval_env=Monitor(CognitiveSingleAPEnv(RadioParams(validate_mode='validation')))
    model=PPO('MlpPolicy',env,learning_rate=3e-4,n_steps=512,
              batch_size=256,n_epochs=10,gamma=.98,gae_lambda=.95,
              clip_range=.2,ent_coef=.01,policy_kwargs={'net_arch':{'pi':[128,128],'vf':[128,128]}},
              verbose=1,seed=seed,device='cpu',tensorboard_log=str(out/'tb'))
    callback=EvalCallback(eval_env,best_model_save_path=str(out/'selection'),
                          log_path=str(out/'validation'),
                          eval_freq=max(1,20000//n_envs),
                          n_eval_episodes=15,deterministic=True,verbose=1)
    model.learn(total_timesteps=timesteps,callback=callback,progress_bar=False)
    model.save(str(out/'last_model'))
    ckpt=out/'selection'/'best_model.zip'
    if not ckpt.is_file():
        raise RuntimeError('Expected best checkpoint was not saved; increase training timesteps')
    return ckpt


def assert_checkpoint_schema(model):
    """Prevents loading old 8D/7D model in a new 12D/3D pipeline."""
    if tuple(model.observation_space.shape)!=(12,):
        raise ValueError(f'Checkpoint obs shape {model.observation_space.shape} != 12')
    if tuple(map(int,model.action_space.nvec))!=(3,3,7):
        raise ValueError(f'Checkpoint action nvec {model.action_space.nvec} != [3,3,7]')


def manifest():
    return {
      'pipeline':'CognitiveWiFi-SingleAP-Synthetic-v2',
      'model_observation_schema':list(OBS12_NAMES),
      'model_observation_normalized_0_to_1':True,
      'action_schema':list(ACTION3_NAMES),
      'action_nvec':list(map(int,ACTION3_NVECS)),
      'supported_hardware_operations_reported_by_author':['channel','bandwidth','txpower'],
      'unverified_hardware_operations':['mcs_ceiling','beacon','obss_pd','beamform'],
      'deployment_status':'SHADOW_ONLY_UNCALIBRATED',
      'required_before_hil':['readback actual applied configuration','calibrate RF/queue model',
                            'collect genuine flow QoE traces','offline replay and shadow validation',
                            'verify channel legality and bounded actions','rollback/guardrails'],
      'legacy_8d_7d_compatible':False,
      'multi_ap_joint_training':False,
      'synthetic_p99_is_not_measured_p99':True,
    }


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--timesteps',type=int,default=360000)
    parser.add_argument('--seeds',type=int,nargs='+',default=[11,22,33])
    parser.add_argument('--out',type=Path,default=Path('runs/cognitive_wifi_v2'))
    parser.add_argument('--test-only',type=Path,default=None)
    args=parser.parse_args()
    args.out.mkdir(parents=True,exist_ok=True)
    PPO,*_=_require_sb3()
    policies={}
    if args.test_only is None:
        for seed in args.seeds:
            p=train_one(seed,args.timesteps,args.out)
            m=PPO.load(str(p),device='cpu');assert_checkpoint_schema(m)
            policies[f'ppo_seed_{seed}']=m
    else:
        m=PPO.load(str(args.test_only),device='cpu');assert_checkpoint_schema(m)
        policies['ppo']=m
    policies={'fixed': 'fixed','heuristic':'heuristic',**policies}
    # Choose policy using ONLY validation. Test is consulted exactly once afterwards.
    val_seeds=range(20000,20040)
    val=evaluate_benchmark(policies,seeds=val_seeds,mode='validation',
                           export=args.out/'validation_episodes.csv')
    val_sum=summarize(val)
    ppo_names=[n for n in policies if n.startswith('ppo')]
    selected=max(ppo_names,key=lambda n:val_sum[n]['reward']['mean'])
    test_policies={'fixed':'fixed','heuristic':'heuristic','selected_ppo':policies[selected]}
    test=evaluate_benchmark(test_policies,seeds=range(90000,90100),mode='ood',
                            export=args.out/'heldout_ood_episodes.csv')
    report={'model_status':'NOT_VALIDATED_FOR_REAL_ROUTER','selected_using_validation':selected,
            'validation':val_sum,'heldout_ood_test':summarize(test),
            'caveat':'Synthetic uncalibrated test; not evidence of operational Wi-Fi gains.'}
    (args.out/'report_v2.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
    (args.out/'model_manifest_v2.json').write_text(json.dumps(manifest(),indent=2,ensure_ascii=False),encoding='utf8')
    print(json.dumps(report,indent=2,ensure_ascii=False))
    if args.test_only is None:
        # Never overwrite/rename original checkpoint. Selection is by validation only.
        import shutil
        src=args.out/f"seed_{selected.split('_')[-1]}"/'selection'/'best_model.zip'
        shutil.copy2(src,args.out/'best_model_v2_UNCALIBRATED.zip')
    print('SHADOW ONLY: do not replace live saved_models/best_model.zip')

if __name__=='__main__':
    main()


## 3. Unit Tests — Đóng băng các bất biến quan trọng

Kiểm tra seed reproducibility, 20/40/80 MHz spectral overlap, action-dependent transition, queue boundedness, schema validation.

In [ ]:
%%writefile test_core_v2.py
"""Pure-NumPy tests; do NOT require Stable-Baselines3 or a router."""
import unittest
import numpy as np
from wifi_rl_core_v2 import (
 SingleAPRadioCore,covered_20mhz,encode_shadow_observation,
 validate_telemetry_snapshot,safe_legacy7_from3,OBS8_NAMES)

class TestCognitiveWiFiCore(unittest.TestCase):
    def test_channel_bonding(self):
        self.assertEqual(covered_20mhz(0,2),covered_20mhz(1,2))
        self.assertEqual(covered_20mhz(1,2),covered_20mhz(2,2))
        self.assertEqual(covered_20mhz(0,1),(36,40))
        self.assertEqual(covered_20mhz(1,1),(36,40))
        self.assertEqual(covered_20mhz(2,1),(44,48))
        self.assertEqual(covered_20mhz(0,0),(36,))
        self.assertEqual(covered_20mhz(2,0),(44,))

    def test_seed_reproducible(self):
        a=SingleAPRadioCore(); b=SingleAPRadioCore()
        a.reset(seed=123); b.reset(seed=123)
        self.assertTrue(np.array_equal(a.observe(),b.observe()))
        for _ in range(24):
            z1,r1,_=a.transition([0,2,3])
            z2,r2,_=b.transition([0,2,3])
            self.assertEqual(z1,z2)
            self.assertEqual(r1,r2)

    def test_state_is_action_dependent(self):
        x=SingleAPRadioCore();y=SingleAPRadioCore()
        x.reset(43); y.reset(43)
        a,r,_=x.transition([0,0,0])
        b,s,_=y.transition([2,2,6])
        # UL station RSSI must NOT be inferred directly from AP TX power.
        self.assertNotEqual(float(a['capacity_mbps']),float(b['capacity_mbps']))
        self.assertNotEqual(r,s)
        self.assertFalse(np.array_equal(x.observe(),y.observe()))

    def test_queue_and_goodput_bounded(self):
        env=SingleAPRadioCore();env.reset(77)
        for i in range(100):
            m,r,_=env.transition([0,0,0])
            self.assertGreaterEqual(m['queue_mbit'],0)
            self.assertLessEqual(m['queue_mbit'],env.params.queue_limit_mbit)
            self.assertGreaterEqual(m['throughput_mbps'],0)
            self.assertLessEqual(m['throughput_mbps'],m['capacity_mbps']+1e-5)
            self.assertTrue(0<=m['loss_rate']<=1)
            self.assertTrue(0<=m['uqs_proxy']<=1)
            self.assertTrue(np.isfinite(r))
            self.assertTrue(np.all(np.isfinite(env.observe())))
            self.assertTrue(np.all((env.observe()>=0)&(env.observe()<=1)))

    def test_bridge(self):
        np.testing.assert_array_equal(safe_legacy7_from3([1,2,5]),[1,2,5,7,1,0,0])
        with self.assertRaises(ValueError):
            safe_legacy7_from3([99,2,1])

    def test_shadow_schema(self):
        snap={k:v for k,v in zip(OBS8_NAMES,[-55,-94,19,.11,.16,.38,.4,0])}
        outcome=validate_telemetry_snapshot(snap)
        self.assertTrue(outcome['shadow_only'])
        self.assertGreater(len(outcome['alerts']),0)
        a=encode_shadow_observation(outcome['observation8'],[0,2,3],2)
        self.assertEqual(a.shape,(12,))
        self.assertTrue(np.all((a>=0)&(a<=1)))
        with self.assertRaises(ValueError):
            validate_telemetry_snapshot(dict(snap,noise_floor=None))

if __name__=='__main__':unittest.main()


In [ ]:
!python -m unittest test_core_v2 -v

## 4. Chạy PPO Training (chỉ khi chủ động bật)

Mặc định `TRAIN_NOW=False` nhằm tránh huấn luyện tốn tài nguyên khi đang xem notebook. Sau khi chạy toàn bộ test, đổi sang `True` để train 3 seed x 360k timesteps. Bản 40k timesteps chỉ dùng test pipeline, **không** phải kết quả có thể công bố.

Checkpoint được lựa chọn bằng validation, **không** lấy benchmark held-out để chọn mô hình.

In [ ]:
import subprocess, sys
TRAIN_NOW = False
TIMESTEPS = 360_000
SEEDS = [11,22,33]
OUT_DIR = 'runs/cognitive_wifi_v2'
if TRAIN_NOW:
    subprocess.run([sys.executable, 'train_evaluate_v2.py',
                    '--timesteps', str(TIMESTEPS), '--seeds', *map(str,SEEDS),
                    '--out', OUT_DIR], check=True)
else:
    print('TRAIN_NOW=False — chưa huấn luyện, chưa có checkpoint mới.')

## 5. Kiểm thử không cần PPO — thấy tác động nhân–quả

Sử dụng cùng một seed để mô phỏng hai hành động khác nhau từ cùng một trạng thái. Khác biệt ở next state chứng minh cấu trúc transition đã được thay đổi. **Không** phải minh chứng khả năng tổng quát hóa ngoài thực tế.

In [ ]:
from wifi_rl_core_v2 import SingleAPRadioCore, covered_20mhz
import numpy as np
x,y=SingleAPRadioCore(),SingleAPRadioCore()
x.reset(42); y.reset(42)
mx,rx,_=x.transition([0,0,0])
my,ry,_=y.transition([2,2,6])
print('Bandwidth coverage HE80: ', [covered_20mhz(i,2) for i in range(3)])
print('HE20/14dBm: ', {k:round(mx[k],3) for k in ('capacity_mbps','latency_p99_ms','loss_rate','queue_mbit')}, 'reward',round(rx,3))
print('HE80/20dBm: ', {k:round(my[k],3) for k in ('capacity_mbps','latency_p99_ms','loss_rate','queue_mbit')}, 'reward',round(ry,3))
assert not np.array_equal(x.observe(),y.observe())

## 6. Shadow-only Adapter

**Cần** đọc lại cấu hình đang áp dụng từ router nếu muốn chạy shadow mode; chưa có dữ liệu thì không được suy đoán. Cell này không thực thi kết nối mạng, không apply action.

In [ ]:
%%writefile shadow_inference_v2.py
"""Inference only. NO network requests, NO SSH, NO router changes."""
import argparse,json
from pathlib import Path
import numpy as np
from wifi_rl_core_v2 import (validate_telemetry_snapshot,encode_shadow_observation,
                            safe_legacy7_from3)


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--snapshot',required=True,type=Path)
    p.add_argument('--checkpoint',required=True,type=Path)
    p.add_argument('--channel-idx',type=int,default=None)
    p.add_argument('--bw-idx',type=int,default=None)
    p.add_argument('--tx-idx',type=int,default=None)
    p.add_argument('--dwell-cycles',type=int,default=0)
    args=p.parse_args()
    from stable_baselines3 import PPO
    from train_evaluate_v2 import assert_checkpoint_schema
    if None in (args.channel_idx,args.bw_idx,args.tx_idx):
        raise ValueError('You MUST provide read-back applied config indexes; never guess current router config')
    snapshot=json.loads(args.snapshot.read_text('utf-8'))
    # If file holds multiple BSSIDs, select one explicitly upstream, never mix APs.
    if not ('avg_rssi' in snapshot):
        raise ValueError('Snapshot must describe exactly ONE AP with raw 8-D fields')
    check=validate_telemetry_snapshot(snapshot)
    obs=encode_shadow_observation(check['observation8'],
                np.array([args.channel_idx,args.bw_idx,args.tx_idx]),args.dwell_cycles)
    model=PPO.load(str(args.checkpoint),device='cpu'); assert_checkpoint_schema(model)
    choice,_=model.predict(obs,deterministic=True)
    print(json.dumps({
        'mode':'SHADOW_ONLY_NOT_FOR_ROUTER_EXECUTION',
        'proposed_action_3d_indices':list(map(int,choice)),
        'legacy_7d_mapping_for_visualization_only':list(map(int,safe_legacy7_from3(choice))),
        'alerts':check['alerts']+['No action applied','No live QoE claims','Uncalibrated simulator'],
    },indent=2))

if __name__=='__main__':main()


## 7. Đọc kết quả benchmark sau khi train

Chỉ báo cáo các giá trị thật sự có trong file xuất ra. Nếu chưa train, cell không tạo hoặc bịa số liệu.

In [ ]:
import json
from pathlib import Path
report_path = Path(OUT_DIR)/'report_v2.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf8'))
    print(json.dumps(report, indent=2, ensure_ascii=False)[:12000])
else:
    print('Chưa có report. Cần chạy Training + Evaluation trước.')

## 8. Các giới hạn chưa được giải quyết bằng phần mềm thuần túy

**Chưa có dữ liệu hiệu chuẩn của testbed**, nên tất cả mô hình PHY rate, contention, queue, synthetic per-packet P99 đều là *phenomenological assumptions*, không được coi là một mô phỏng chuẩn IEEE 802.11 đã được kiểm chứng. `ofdma_ratio` có thể là proxy trong telemetry OpenWrt, không phải tỷ lệ OFDMA đo trực tiếp. Không dùng đầu ra synthetic để tuyên bố QoE trên phần cứng đã được bảo đảm.

**Giai đoạn kế tiếp trong Yêu cầu 1:** Dùng log thật để hiệu chuẩn, kiểm tra mô hình trên held-out real-world traces, xác minh interface chính xác và chạy shadow/HIL dưới sự giám sát. **Chưa chuyển sang Yêu cầu 2 Multi-AP**.